In [ ]:
import torch
import torch.nn as nn

class SimpleRNN(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(SimpleRNN, self).__init__()
        self.rnn = nn.RNN(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, _ = self.rnn(x)          # RNN output
        out = out[:, -1, :]           # Last time step
        out = self.fc(out)
        return out

# Example usage
model = SimpleRNN(input_size=10, hidden_size=20, output_size=2)
x = torch.randn(5, 3, 10)  # (batch, seq_len, input_size)
output = model(x)
print(output)

tensor([[ 0.1140, -0.2088],
        [-0.1685, -0.1188],
        [ 0.3613, -0.2496],
        [-0.1917, -0.3900],
        [ 0.3896,  0.0365]], grad_fn=<AddmmBackward0>)


In [ ]:
import torch
import torch.nn as nn

class LSTMModel(nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super(LSTMModel, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)

    def forward(self, x):
        out, (hn, cn) = self.lstm(x)
        out = out[:, -1, :]
        out = self.fc(out)
        return out

# Example
model = LSTMModel(10, 20, 2)
x = torch.randn(5, 3, 10)
output = model(x)
print(output)

tensor([[ 0.1385, -0.0771],
        [ 0.0240, -0.1075],
        [ 0.1396,  0.0012],
        [ 0.1203,  0.0074],
        [-0.0276, -0.0436]], grad_fn=<AddmmBackward0>)


In [ ]:
# Install if not already
# !pip install transformers datasets torch

import torch
from transformers import BertTokenizer, BertForSequenceClassification
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F

# -----------------------------
# 1. Dummy Dataset (for training)
# -----------------------------
class SimpleDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=32):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts[idx],
            padding='max_length',
            truncation=True,
            max_length=self.max_len,
            return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(),
            "attention_mask": encoding["attention_mask"].squeeze(),
            "labels": torch.tensor(self.labels[idx])
        }

# Sample data
texts = [
    "I love this product",
    "This is amazing",
    "I hate this",
    "Very bad experience"
]
labels = [1, 1, 0, 0]  # 1 = positive, 0 = negative

# -----------------------------
# 2. Load Model & Tokenizer
# -----------------------------
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
model = BertForSequenceClassification.from_pretrained(
    'bert-base-uncased',
    num_labels=2
)

# -----------------------------
# 3. DataLoader
# -----------------------------
dataset = SimpleDataset(texts, labels, tokenizer)
loader = DataLoader(dataset, batch_size=2, shuffle=True)

# -----------------------------
# 4. Training Setup
# -----------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=2e-5)

# -----------------------------
# 5. Training Loop
# -----------------------------
model.train()

for epoch in range(3):  # small epochs
    print(f"Epoch {epoch+1}")
    for batch in loader:
        optimizer.zero_grad()

        outputs = model(
            input_ids=batch["input_ids"],
            attention_mask=batch["attention_mask"],
            labels=batch["labels"]
        )

        loss = outputs.loss
        loss.backward()
        optimizer.step()

        print("Loss:", loss.item())

# -----------------------------
# 6. Evaluation / Prediction
# -----------------------------
model.eval()

test_text = "I really love this!"
inputs = tokenizer(test_text, return_tensors="pt")

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# Convert to probabilities
probs = F.softmax(logits, dim=1)

# Predicted class
pred = torch.argmax(probs, dim=1)

print("\nText:", test_text)
print("Logits:", logits)
print("Probabilities:", probs)
print("Predicted class:", pred.item())

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch 1
Loss: 0.7489526867866516
Loss: 0.6971296072006226
Epoch 2
Loss: 0.6315114498138428
Loss: 0.6241562962532043
Epoch 3
Loss: 0.5347212553024292
Loss: 0.5854474306106567

Text: I really love this!
Logits: tensor([[-0.0101,  0.8130]])
Probabilities: tensor([[0.3051, 0.6949]])
Predicted class: 1


In [ ]:
from transformers import RobertaTokenizer, RobertaForSequenceClassification

# Load model
tokenizer = RobertaTokenizer.from_pretrained('roberta-base')
model = RobertaForSequenceClassification.from_pretrained('roberta-base')

# Sample input
text = "Deep learning is powerful"
inputs = tokenizer(text, return_tensors="pt")

# Forward pass
outputs = model(**inputs)
logits = outputs.logits

print(logits)

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: roberta-base
Key                             | Status     | 
--------------------------------+------------+-
lm_head.bias                    | UNEXPECTED | 
roberta.embeddings.position_ids | UNEXPECTED | 
lm_head.dense.bias              | UNEXPECTED | 
lm_head.layer_norm.weight       | UNEXPECTED | 
lm_head.dense.weight            | UNEXPECTED | 
lm_head.layer_norm.bias         | UNEXPECTED | 
classifier.out_proj.weight      | MISSING    | 
classifier.dense.weight         | MISSING    | 
classifier.dense.bias           | MISSING    | 
classifier.out_proj.bias        | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


tensor([[-0.0082,  0.0586]], grad_fn=<AddmmBackward0>)
